In [ ]:
import numpy as np
from reachy_sdk import ReachySDK
from reachy_sdk.trajectory import goto
from reachy_sdk.trajectory.interpolation import InterpolationMode

reachy = ReachySDK(host="10.22.128.166")

In [ ]:
reachy

In [ ]:
import wave
import json
from vosk import Model, KaldiRecognizer

MODEL_PATH = "vosk/vosk-model-small-en-us-0.15"
WAV_FILE = "command.wav"

model = Model(MODEL_PATH)

wf = wave.open(WAV_FILE, "rb")

print("Channels:", wf.getnchannels())
print("Rate:", wf.getframerate())
print("Width:", wf.getsampwidth())

COMMAND_WORDS = [
    "reachy",
    "move",
    "pick",
    "up",
    "put",
    "place",
    "cube",
    "cylinder",
    "empty",
    "square",
    "stop",
    "[unk]"
]

recognizer = KaldiRecognizer(
    model,
    wf.getframerate(),
    json.dumps(COMMAND_WORDS)
)


while True:
    data = wf.readframes(4000)

    if len(data) == 0:
        break

    if recognizer.AcceptWaveform(data):
        result = json.loads(
            recognizer.Result()
        )

        if result.get("text"):
            print("Heard:", result["text"])

final_result = json.loads(
    recognizer.FinalResult()
)

print("Final:", final_result.get("text", ""))

In [ ]:
import subprocess
import numpy as np
import json
from vosk import Model, KaldiRecognizer

MODEL_PATH = "vosk/vosk-model-small-en-us-0.15"

model = Model(MODEL_PATH)

words = [
    "move",
    "cube",
    "cylinder",
    "stop",
    "[unk]"
]

recognizer = KaldiRecognizer(
    model,
    16000,
    json.dumps(words)
)

# Capture raw 6-channel audio from ReSpeaker
process = subprocess.Popen(
    [
        "arecord",
        "-D", "hw:0,0",
        "-f", "S16_LE",
        "-r", "16000",
        "-c", "6",
        "-t", "raw"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.DEVNULL
)

print("Listening... Say 'move cube', 'move cylinder', or 'stop'.")

while True:

    # 4000 samples × 6 channels × 2 bytes
    data = process.stdout.read(4000 * 6 * 2)

    if not data:
        break

    # Convert raw interleaved audio to numpy
    audio = np.frombuffer(data, dtype=np.int16)

    # Separate six channels
    audio = audio.reshape(-1, 6)

    # Use microphone channel 0
    mono = audio[:, 0].astype(np.int16)

    if recognizer.AcceptWaveform(mono.tobytes()):

        result = json.loads(recognizer.Result())
        text = result.get("text", "")

        if text:
            print("Heard:", text)

            if "stop" in text:
                print("COMMAND: STOP")
                break

            elif "move" in text and "cube" in text:
                print("COMMAND: MOVE CUBE")

            elif "move" in text and "cylinder" in text:
                print("COMMAND: MOVE CYLINDER")

process.terminate()
print("Stopped.")